# Exploratory Data Analysis (EDA) for Fertilizer Prediction

This notebook focuses on understanding the dataset for fertilizer prediction through various exploratory data analysis techniques.
We will:
1. Load the dataset.
2. Perform initial inspection (info, describe, null values).
3. Analyze the target variable ('Fertilizer Name').
4. Conduct univariate analysis of numerical and categorical features.
5. Conduct bivariate analysis to explore relationships between features and the target variable, and between features themselves.

## 1. Setup and Data Loading

### 1.1 Import Libraries and Configure Settings

This cell imports all the standard libraries required for data manipulation, visualization, and sets up global configurations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import kagglehub # For downloading dataset if needed

# Configure settings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 7) # Default figure size

### 1.2 Download Dataset from Kaggle (if not already present)

This step ensures the dataset is available. It uses `kagglehub` to download the 'fertilizer-prediction' dataset. The actual data loading will expect standard Kaggle paths.

In [ ]:
# Path where Kaggle datasets are typically downloaded or can be expected
dataset_path_root = '/kaggle/input/'
fertilizer_dataset_path = os.path.join(dataset_path_root, 'fertilizer-prediction', 'Fertilizer Prediction.csv')
playground_dataset_path = os.path.join(dataset_path_root, 'playground-series-s5e6', 'train.csv')

if not (os.path.exists(fertilizer_dataset_path) and os.path.exists(playground_dataset_path)):
    print("Dataset files not found at standard Kaggle paths. Attempting download...")
    try:
        # This will download to a path like ~/.cache/kagglehub/datasets/irakozekelly/fertilizer-prediction
        # For the notebook to use it directly, you might need to adjust paths or ensure it's in /kaggle/input
        path_downloaded = kagglehub.dataset_download("irakozekelly/fertilizer-prediction")
        print(f"Dataset downloaded to: {path_downloaded}")
        # Note: The user might need to manually move/link this to /kaggle/input for the subsequent cells to work as is.
        # For now, we'll assume the primary playground dataset is the one to focus on.
        # playground_path_downloaded = kagglehub.dataset_download("competitions/playground-series-s5e6/playground-series-s5e6")
        # print(f"Playground Series S5E6 data downloaded to: {playground_path_downloaded}") 
        print("Please ensure the downloaded files are accessible via /kaggle/input/ for the next steps.")
    except Exception as e:
        print(f"Could not download dataset: {e}")
else:
    print("Dataset files found at standard Kaggle paths.")

### 1.3 Load and Inspect Training Data

The primary dataset for training our model is loaded from `/kaggle/input/playground-series-s5e6/train.csv`. We then perform an initial inspection.

In [ ]:
# Load the dataset
df = pd.DataFrame() # Initialize an empty dataframe
try:
    # Primary dataset for this task as per original notebook context
    df = pd.read_csv('/kaggle/input/playground-series-s5e6/train.csv') 
    if 'id' in df.columns:
        df = df.drop('id', axis=1) # Drop 'id' if it exists
    print("Dataset '/kaggle/input/playground-series-s5e6/train.csv' loaded successfully.")
except FileNotFoundError:
    print("Error: '/kaggle/input/playground-series-s5e6/train.csv' not found.")
    print("Attempting to load from an alternative common path for 'Fertilizer Prediction.csv'...")
    try:
        # Fallback to the other dataset if the primary one isn't found (adjust as needed)
        df = pd.read_csv('/kaggle/input/fertilizer-prediction/Fertilizer Prediction.csv')
        print("Dataset '/kaggle/input/fertilizer-prediction/Fertilizer Prediction.csv' loaded successfully.")
    except FileNotFoundError:
        print("Error: Also failed to load '/kaggle/input/fertilizer-prediction/Fertilizer Prediction.csv'.")
        print("Please ensure your dataset is correctly placed in the /kaggle/input/ directory.")

if not df.empty:
    print("\nFirst 5 rows of the dataset:")
    display(df.head())
    print("\nDataset Information:")
    df.info()
    print("\nDescriptive Statistics:")
    display(df.describe())
    print("\nMissing values:")
    display(df.isnull().sum())
else:
    print("\nDataset could not be loaded. Cannot proceed with EDA.")

The column 'Humidity ' (with a trailing space) is often present. Let's rename it for consistency if it exists.

In [ ]:
if not df.empty:
    if 'Humidity ' in df.columns:
        df.rename(columns={'Humidity ': 'Humidity'}, inplace=True)
        print("Renamed 'Humidity ' to 'Humidity'.")
        print("Updated columns:", df.columns.tolist())
    else:
        print("'Humidity ' column with trailing space not found.")
else:
    print("DataFrame is empty, skipping column rename.")

## 2. Exploratory Data Analysis (EDA)

In this section, we'll explore the data to understand its characteristics and relationships between variables.

### 2.1 Target Variable Analysis: 'Fertilizer Name'
Understanding the distribution of the target variable is crucial. We'll check the counts of each fertilizer type.

In [ ]:
if not df.empty and 'Fertilizer Name' in df.columns:
    print("Distribution of Fertilizer Name:")
    fertilizer_counts = df['Fertilizer Name'].value_counts()
    print(fertilizer_counts)
    
    plt.figure(figsize=(12, 7))
    sns.barplot(x=fertilizer_counts.index, y=fertilizer_counts.values, palette='viridis')
    plt.title('Distribution of Fertilizer Name')
    plt.xlabel('Fertilizer Name')
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
elif df.empty:
    print("DataFrame is empty, skipping target variable analysis.")
else:
    print("Target column 'Fertilizer Name' not found in DataFrame.")

**Observation (based on typical data):** 
The dataset is usually well-balanced across the different fertilizer types. Some classes might be slightly more or less frequent, but major class imbalance is not typically observed. This is a good starting point for classification.

### 2.2 Univariate Analysis: Numerical Features

Let's examine the distribution of each numerical feature using histograms and box plots.

In [ ]:
if not df.empty:
    # Infer numerical features (excluding any known target if it was accidentally numeric)
    numerical_features = df.select_dtypes(include=np.number).columns.tolist()
    # If 'Fertilizer Name_Encoded' or similar target is present, remove it
    if 'Fertilizer Name_Encoded' in numerical_features:
        numerical_features.remove('Fertilizer Name_Encoded')
    # Also remove if original dataset had other ID or target like columns that are numeric
    # This list is based on the common features in the dataset:
    expected_numerical_features = ['Temparature', 'Humidity', 'Moisture', 'Nitrogen', 'Potassium', 'Phosphorous']
    numerical_features = [f for f in expected_numerical_features if f in df.columns and df[f].dtype in [np.number, 'int64', 'float64']]

    if numerical_features:
        print(f"Identified numerical features for analysis: {numerical_features}")
        # Histograms
        df[numerical_features].hist(bins=20, figsize=(15, 10), layout=(2, 3), color='skyblue')
        plt.suptitle('Histograms of Numerical Features', fontsize=16, y=1.02)
        plt.tight_layout(rect=[0, 0, 1, 0.98])
        plt.show()
        
        # Box plots
        plt.figure(figsize=(15, 10))
        for i, col in enumerate(numerical_features):
            plt.subplot(2, 3, i + 1)
            sns.boxplot(y=df[col], color='lightcoral')
            plt.title(col)
        plt.suptitle('Box Plots of Numerical Features', fontsize=16, y=1.02)
        plt.tight_layout(rect=[0, 0, 1, 0.98])
        plt.show()
    else:
        print("No numerical features found or defined for analysis.")
else:
    print("DataFrame is empty, skipping numerical feature analysis.")

**Observation (based on typical data):**
The numerical features often exhibit distributions that are somewhat uniform or slightly skewed. Box plots help identify the spread and potential outliers for each of these features.

### 2.3 Univariate Analysis: Categorical Features

Let's examine the distribution of 'Soil Type' and 'Crop Type'.

In [ ]:
if not df.empty:
    expected_categorical_features = ['Soil Type', 'Crop Type']
    categorical_features = [f for f in expected_categorical_features if f in df.columns and df[f].dtype == 'object']

    if categorical_features:
        print(f"Identified categorical features for analysis: {categorical_features}")
        plt.figure(figsize=(18, 6))
        for i, col in enumerate(categorical_features):
            plt.subplot(1, len(categorical_features), i + 1)
            sns.countplot(data=df, y=col, order=df[col].value_counts().index, palette='pastel')
            plt.title(f'Distribution of {col}')
            plt.xlabel('Count')
            plt.ylabel(col)
        plt.tight_layout()
        plt.show()
    else:
        print("No categorical features found or defined for analysis.")
else:
    print("DataFrame is empty, skipping categorical feature analysis.")

**Observation (based on typical data):**
* 'Soil Type' usually has a few distinct categories (e.g., Sandy, Loamy, Clayey, Red, Black), with varying frequencies.
* 'Crop Type' typically has more categories, representing different crops (e.g., Maize, Sugarcane, Cotton, Paddy), also with varying frequencies.

### 2.4 Bivariate Analysis

Now, let's explore relationships between features and the target variable ('Fertilizer Name').

#### 2.4.1 Numerical Features vs. Target ('Fertilizer Name')
We use box plots and violin plots to see how numerical feature distributions vary across different fertilizer types.

In [ ]:
if not df.empty and 'Fertilizer Name' in df.columns and numerical_features:
    print("Generating Box Plots for Numerical Features vs. Fertilizer Name...")
    plt.figure(figsize=(20, max(25, len(numerical_features)*5))) # Adjust height based on num features
    for i, col in enumerate(numerical_features):
        plt.subplot(len(numerical_features), 1, i + 1)
        sns.boxplot(data=df, x='Fertilizer Name', y=col, palette='Set3')
        plt.title(f'{col} by Fertilizer Name')
        plt.xticks(rotation=45, ha='right')
    plt.suptitle('Numerical Features vs. Fertilizer Name (Box Plots)', fontsize=18, y=1.0)
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    plt.show()

    print("\nGenerating Violin Plots for Numerical Features vs. Fertilizer Name...")
    plt.figure(figsize=(20, max(25, len(numerical_features)*5)))
    for i, col in enumerate(numerical_features):
        plt.subplot(len(numerical_features), 1, i + 1)
        sns.violinplot(data=df, x='Fertilizer Name', y=col, palette='muted')
        plt.title(f'{col} by Fertilizer Name')
        plt.xticks(rotation=45, ha='right')
    plt.suptitle('Numerical Features vs. Fertilizer Name (Violin Plots)', fontsize=18, y=1.0)
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    plt.show()
elif df.empty:
    print("DataFrame is empty, skipping numerical features vs. target analysis.")
elif 'Fertilizer Name' not in df.columns:
    print("Target 'Fertilizer Name' not in DataFrame for bivariate analysis.")
else:
    print("Numerical features not identified for bivariate analysis.")

**Observations (based on typical data):**
* **Nutrient Content (N, P, K):** These are typically strong differentiators. For example, 'Urea' is associated with high Nitrogen. 'DAP' (Di-Ammonium Phosphate) would show higher N and P. Fertilizers like '10-26-26' would show specific levels for N, P, and K respectively.
* **Environmental Factors (Temperature, Humidity, Moisture):** Certain fertilizers might be preferred under specific environmental conditions, which these plots can help reveal.

#### 2.4.2 Categorical Features vs. Target ('Fertilizer Name')
We use count plots (grouped bar charts) and stacked bar charts to see how fertilizer types are distributed across different soil and crop types.

In [ ]:
if not df.empty and 'Fertilizer Name' in df.columns and categorical_features:
    print("Generating Count Plots for Categorical Features vs. Fertilizer Name...")
    for col in categorical_features:
        plt.figure(figsize=(14, 8))
        sns.countplot(data=df, y='Fertilizer Name', hue=col, palette='Spectral', order = df['Fertilizer Name'].value_counts().index)
        plt.title(f'Fertilizer Name Distribution by {col}')
        plt.xlabel('Count')
        plt.ylabel('Fertilizer Name')
        plt.legend(title=col, bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()
        
    print("\nGenerating Stacked Bar Charts for Categorical Features vs. Fertilizer Name...")
    for col in categorical_features:
        # Crosstab to get counts for stacking
        cross_tab = pd.crosstab(df[col], df['Fertilizer Name'])
        # Normalize to get proportions for 100% stacked bar chart
        cross_tab_prop = cross_tab.apply(lambda x: x / x.sum() * 100, axis=1)
        
        cross_tab_prop.plot(kind='bar', stacked=True, colormap='viridis', figsize=(14,10))
        plt.title(f'Proportional Distribution of Fertilizer Name by {col}')
        plt.xlabel(col)
        plt.ylabel('Percentage')
        plt.xticks(rotation=45, ha='right')
        plt.legend(title='Fertilizer Name', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()
elif df.empty:
    print("DataFrame is empty, skipping categorical features vs. target analysis.")
elif 'Fertilizer Name' not in df.columns:
    print("Target 'Fertilizer Name' not in DataFrame for bivariate analysis.")
else:
    print("Categorical features not identified for bivariate analysis.")

**Observations (based on typical data):**
These plots often show strong associations. For example, certain crop types might predominantly use specific fertilizers, or particular soil types might necessitate certain fertilizer compositions. Stacked bar charts are excellent for seeing the proportional distribution.

#### 2.4.3 Pair Plot of Numerical Features
A pair plot helps visualize pairwise relationships between numerical features, colored by the target variable. This can reveal clusters and correlations.

In [ ]:
if not df.empty and 'Fertilizer Name' in df.columns and numerical_features:
    # To avoid overly large plots if there are too many unique fertilizer names for hue,
    # consider sampling or a subset of features if needed, though typically it's manageable.
    print("Generating Pair Plot of Numerical Features by Fertilizer Name...")
    
    # Make sure 'Fertilizer Name' is included for hue
    plot_df = df[numerical_features + ['Fertilizer Name']].copy()
    
    # Reduce complexity if too many categories for hue in pairplot to prevent clutter
    # For example, if more than 10 fertilizer types, this might become very dense.
    # However, the original notebook plotted them all, so we will attempt the same.
    
    plt.figure(figsize=(12,12)) # Ensure figure size is adequate
    sns.pairplot(plot_df, hue='Fertilizer Name', palette='tab10', diag_kind='kde') # Using kde for diagonal
    plt.suptitle('Pair Plot of Numerical Features by Fertilizer Name', y=1.02)
    plt.show()
elif df.empty:
    print("DataFrame is empty, skipping pair plot.")
elif 'Fertilizer Name' not in df.columns:
    print("Target 'Fertilizer Name' not in DataFrame for pair plot hue.")
else:
    print("Numerical features not identified for pair plot.")

**Observations (based on typical data):**
- The pair plot can show how different fertilizer types cluster in the multi-dimensional space of numerical features.
- Clear separations for some fertilizers based on N, P, K values are often visible.
- Relationships between features like Temperature and Humidity might also be observed.

#### 2.4.4 Correlation Matrix of Numerical Features
A heatmap of the correlation matrix shows linear relationships between numerical features.

In [ ]:
if not df.empty and numerical_features:
    print("Generating Correlation Matrix of Numerical Features...")
    correlation_matrix = df[numerical_features].corr()
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=.5)
    plt.title('Correlation Matrix of Numerical Features')
    plt.show()
elif df.empty:
    print("DataFrame is empty, skipping correlation matrix.")
else:
    print("Numerical features not identified for correlation matrix.")

**Observations (based on typical data):**
- Moderate correlations might be observed between some nutrient pairs (e.g., Potassium and Phosphorous).
- Temperature and Humidity might show some correlation.
- Generally, multicollinearity among the original numerical features is not a major issue, but this plot helps verify that.

## 3. EDA Summary

This EDA has provided insights into:
- The distribution of the target variable ('Fertilizer Name') and individual features.
- Relationships between input features and the target variable.
- Correlations among numerical features.

Key takeaways often include:
- Nutrient levels (N, P, K) are strong predictors.
- Soil type and crop type have significant interactions with fertilizer choice.
- The dataset is generally clean with no missing values and relatively balanced classes.

These findings will inform the subsequent data preprocessing, feature engineering, and model selection steps.